# RetinoCheck AI — Colab
Research only. Never a medical device or medical advice. Choose a T4 GPU runtime before running. Upload the repository to `MyDrive/RetinoCheckAI/repo`. All project logic is imported from `src/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/RetinoCheckAI/repo
%pip install -q -r requirements.txt

If pip upgraded an already-imported package, restart the session and rerun setup. The download/cache stays on Colab local disk; checkpoints go to mounted Drive.

In [ ]:
import torch
from src.config import Config
from src.data import load_source
from src.train import train
from src.evaluate import evaluate
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — select a T4 runtime")
MODEL = "resnet18"  # Or "efficientnet_b0"; each model gets its own folder.
config = Config(model_name=MODEL, output_dir=f"/content/drive/MyDrive/RetinoCheckAI/{MODEL}")

Verify a real Hugging Face download, schema, labels and one decoded image. Failure raises an actionable error. Fallback: download the original Kaggle APTOS 2019 data, extract `train.csv` and `train_images/`, then set `config.local_data = "/content/aptos2019"` and choose a distinct output directory.

In [ ]:
source, labels, identity = load_source(config)
print("Loaded images:", len(labels), "fingerprint:", identity)
print("Grade counts:", {grade: int((labels == grade).sum()) for grade in range(5)})
del source

Train or resume. After disconnecting, rerun setup and configuration, set `RESUME = True`, and rerun this cell. A partial epoch restarts from the previous completed epoch. Do not reuse a run folder for a different experiment.

In [ ]:
RESUME = False
history = train(config, resume=f"{config.output_dir}/last.pt" if RESUME else None)

Save package versions alongside the run. Compare validation histories between models and choose settings before accessing test results.

In [ ]:
import subprocess
from pathlib import Path
Path(config.output_dir, "environment.txt").write_text(subprocess.check_output(["pip", "freeze"], text=True))

Final held-out evaluation: run once for the model selected using validation results. Report all metrics, class counts and splitting limitations.

In [ ]:
results = evaluate(f"{config.output_dir}/best.pt", output_dir=f"{config.output_dir}/reports")
print(results)

Optional research demo. Running this cell creates a temporary public Gradio link because Colab cannot expose a localhost UI directly. Use public or de-identified research images. Close the demo/runtime after use.

In [ ]:
from app.gradio_app import create_app
demo = create_app(f"{config.output_dir}/best.pt")
demo.launch(share=True)